# Case study: Dual-Level Retrieval ALONE vs. Baseline Hybrid RAG

Variant of `case_study_dual_level_vs_hybrid.ipynb` with the Hybrid RAG contribution removed from the Dual-Level side entirely -- same question, same judge prompt/code, unchanged. Compares:

- **Baseline** -- Hybrid RAG only (`run_hybrid_only_question.py`): BM25 + dense vector over the full Azure AI Search index, no knowledge graph involved.
- **Dual-Level Retrieval ALONE** -- LightRAG local (1-hop neighborhood) + global (semantic relation) retrieval, with NO Hybrid RAG fused in at all (`run_dual_level_only_question.py`) -- isolates what Dual-Level Retrieval contributes entirely on its own, rather than the fused combination production actually uses for the `aggregation` route.

Scored on the same 5 custom dimensions (not ragas's built-in metrics):

| Metric | What it evaluates | Why it matters |
|---|---|---|
| **Completeness** | Does the answer address all aspects of the question? | Multi-aspect engineering questions |
| **Retrieval Coverage** | Does retrieval gather all necessary supporting evidence? | Measures retrieval effectiveness |
| **Groundedness** | Are all claims supported by retrieved documents? | Prevents hallucination |
| **Relevance** | Is the answer focused on the requested engineering problem? | Avoids unnecessary information |
| **Cross-document Integration** | Does the answer successfully combine evidence from multiple documents? | Highlights the advantage of Dual-Level Retrieval |

Question under study (same as the other case study, for direct comparability):

> What are the height-based requirements and spill control considerations for the Air-Cooled Condenser (ACC) in the tender context?

The judge uses this app's existing APIM-authenticated LLM client (`src.pipeline.clients.call_json`), same as `_llm_judge.py` -- not a separate `OPENAI_API_KEY` setup, since this is a custom judge in that same family, not a ragas metric.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any, Dict, List

CHATBOT_ROOT = Path.cwd().resolve()
for _candidate in [CHATBOT_ROOT, *CHATBOT_ROOT.parents]:
    if (_candidate / "src" / "pipeline").exists():
        CHATBOT_ROOT = _candidate
        break
if str(CHATBOT_ROOT) not in sys.path:
    sys.path.insert(0, str(CHATBOT_ROOT))
print("CHATBOT_ROOT =", CHATBOT_ROOT)

from pydantic import BaseModel

from src.pipeline.clients import call_json, validate_model
from run_hybrid_only_question import run_hybrid_only
from run_dual_level_only_question import run_dual_level_only

CHATBOT_ROOT = /Users/senghok/Projects/gds_tender_test/chatbot


/opt/anaconda3/envs/gds_test/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
QUESTION = (
    "Summarize all engineering, safety, and environmental considerations related to the Air-Cooled Condenser (ACC)."
)

## Judge: 5-dimension case-study scorer

One LLM call scores all 5 dimensions together (same pattern as `_llm_judge.py`'s 4-dimension judge -- cheaper and more consistent than 5 separate calls). Each dimension's instructions spell out exactly what distinguishes it from the others -- especially **completeness** (answer-level) vs. **retrieval_coverage** (context-level), which are easy to conflate but measure different pipeline stages.

In [3]:
class CaseStudyScore(BaseModel):
    completeness: int
    retrieval_coverage: int
    groundedness: int
    relevance: int
    cross_document_integration: int
    reasoning: str = ""


CASE_STUDY_JUDGE_SYS = (
    "You are an impartial evaluator comparing two retrieval-augmented generation (RAG) "
    "architectures for a tender/ITB (Invitation to Bid) engineering question-answering "
    "assistant: Hybrid RAG (a baseline that searches an Azure AI Search index with BM25 + "
    "dense vector retrieval) versus Dual-Level Retrieval (a knowledge-graph method that "
    "retrieves via LightRAG's local 1-hop neighborhood and global semantic relation channels, "
    "fused with Hybrid RAG). You score ONE answer to ONE question in isolation, using ONLY the "
    "retrieved contexts provided -- do not use outside knowledge of the true facts, and do not "
    "compare this answer to any other answer. Return strict JSON only."
)

CASE_STUDY_JUDGE_USER = """Question:
{question}

Retrieved contexts (the evidence available to the system when it generated the answer,
one per source file/chunk):
{contexts_block}

Distinct source files represented in the retrieved contexts above: {file_list}

Answer:
{answer}

Score the answer on each dimension below, 1-5 (5 = best). Base every score ONLY on the
question, the retrieved contexts, and the answer given -- not on outside/expert knowledge
of the true facts, since your job is to judge THIS pipeline's behavior, not fact-check it
against ground truth you don't have.

- completeness (1-5): Does the ANSWER address EVERY distinct aspect the question asks
  about? Multi-aspect questions (e.g. "X and Y considerations") require the answer to
  cover ALL named aspects, not just the first or easiest one. 5 = every aspect is
  addressed; 1 = the answer only addresses a single aspect while ignoring others the
  question explicitly asked about.

- retrieval_coverage (1-5): Do the RETRIEVED CONTEXTS ABOVE (not the answer) contain
  enough evidence to fully address every aspect of the question? This scores the
  RETRIEVAL step, not the generation step -- score this LOW if the contexts are missing
  evidence for one of the question's aspects, even if the answer sounds confident; score
  this HIGH if the contexts genuinely contain what's needed, regardless of whether the
  answer actually used it well (that's what completeness measures instead).

- groundedness (1-5): Is every factual claim in the answer directly supported by the
  retrieved contexts? 5 = no claim goes beyond what the contexts state; 1 = the answer
  contains claims that are fabricated, unsupported, or contradicted by the contexts. Do
  not penalize reasonable synthesis or paraphrasing of what the contexts actually say --
  only penalize claims the contexts do not support at all.

- relevance (1-5): Is the answer tightly focused on the engineering problem the question
  actually asks, without padding, tangents, or unrequested information?

- cross_document_integration (1-5): Does the answer meaningfully COMBINE evidence from
  MULTIPLE DISTINCT SOURCE FILES -- not just cite several files trivially, but actually
  synthesize facts that individually come from different documents into one coherent
  answer? 1 = the answer only draws on a single file, or multiple files were available but
  the answer ignores that and only uses one; 5 = the answer genuinely integrates
  information across two or more of the distinct files listed above into a unified answer,
  where that cross-document synthesis is essential (not incidental) to answering the
  question. If only one distinct file is listed above, this dimension cannot be
  demonstrated regardless of the answer -- score it 1 and say so in the reasoning, rather
  than penalizing or crediting the answer for something retrieval never made possible.

Return STRICT JSON only:
{{"completeness": <1-5>, "retrieval_coverage": <1-5>, "groundedness": <1-5>, "relevance": <1-5>, "cross_document_integration": <1-5>, "reasoning": "<2-4 sentences justifying each score, referencing specific evidence gaps or integration points>"}}
"""


def _contexts_block(contexts: List[Dict[str, Any]]) -> str:
    lines = []
    for i, c in enumerate(contexts, 1):
        fn = c.get("file_name", "Unknown")
        pn = c.get("page_number", "N/A")
        content = (c.get("content") or "").strip()[:800]
        lines.append(f"[{i}] {fn} (p.{pn})\n{content}")
    return "\n\n".join(lines) if lines else "(no contexts retrieved)"


def judge_case_study(question: str, contexts: List[Dict[str, Any]], answer: str) -> CaseStudyScore:
    file_list = sorted({c.get("file_name", "Unknown") for c in contexts})
    user_prompt = CASE_STUDY_JUDGE_USER.format(
        question=question,
        contexts_block=_contexts_block(contexts),
        file_list=file_list,
        answer=answer or "(no answer)",
    )
    data = call_json(CASE_STUDY_JUDGE_SYS, user_prompt)
    return validate_model(data, CaseStudyScore)

## Run both pipelines on the same question

In [6]:
print("Running Hybrid-RAG-only baseline...")
baseline_result = run_hybrid_only(QUESTION)
baseline_answer = baseline_result["answer"]
baseline_contexts = baseline_result["reranked_chunks"]
print(f"  {len(baseline_contexts)} context chunks, {len(set(c.get('file_name') for c in baseline_contexts))} distinct files")
print(f"  Answer: {baseline_answer.answer[:300]}")

Running Hybrid-RAG-only baseline...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5624.75it/s]


  20 context chunks, 11 distinct files
  Answer: ### 📝 Engineering Considerations
- **Material Selection**: Materials must be chosen to support piping and equipment design for all operational scenarios, manage corrosion throughout the facility's design life, and ensure thermal energy retention, acoustic hazard management, and exclusion of environm


In [4]:
print("Running Dual-Level Retrieval ALONE (no Hybrid RAG)...")
dual_result = run_dual_level_only(QUESTION)
dual_answer = dual_result["answer"]
dual_contexts = dual_result["reranked_chunks"]
print(f"  {len(dual_contexts)} context chunks, {len(set(c.get('file_name') for c in dual_contexts))} distinct files")
print(f"  Answer: {dual_answer.answer[:300]}")

Running Dual-Level Retrieval ALONE (no Hybrid RAG)...


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=13, offset=13>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 13, 'line': 2, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n            CALL db.index.vector.queryNodes('relation_embedding_index', $query_k, $qvec)\n            YIELD node AS r, score\n            WHERE ($domains IS NULL OR size([(r)-[:CLASSIFIED_AS]->(dom) WHERE dom.domainName IN $domains | dom]) > 0)\n            MATCH (s:Entity)-[:SUBJECT_OF]->(r)-[:OBJECT_OF]->(o:Entity)\n            OPTIO

  45 context chunks, 12 distinct files
  Answer: ### 📝 Engineering Considerations
- **Hot Air Recirculation (HAR):** The ACC must be installed at an appropriate height to minimize HAR risks, which can impact performance. Computational Fluid Dynamics (CFD) studies are recommended to assess and mitigate HAR risks [Source 1].
- **Structural Design:**


In [5]:
print(dual_answer.answer)

### 📝 Engineering Considerations
- **Hot Air Recirculation (HAR):** The ACC must be installed at an appropriate height to minimize HAR risks, which can impact performance. Computational Fluid Dynamics (CFD) studies are recommended to assess and mitigate HAR risks [Source 1].
- **Structural Design:** The ACC structure must account for operational loads, transportation loads, and environmental exceedance events. It should also ensure proper load transfer to foundations and provide suitable access for maintenance [Source 2].
- **Material Selection:** Materials must be selected to manage corrosion and ensure thermal energy retention or exclusion. Insulation systems should also manage acoustic hazards and exclude environmental moisture [Source 2].
- **Process Design:** The ACC must integrate seamlessly with the overall plant design, ensuring operability during normal, start-up, shutdown, and upset conditions [Source 3].

### ✅ Safety Considerations
- **Emergency Response:** The design must 

In [7]:
print(baseline_answer.answer)

### 📝 Engineering Considerations
- **Material Selection**: Materials must be chosen to support piping and equipment design for all operational scenarios, manage corrosion throughout the facility's design life, and ensure thermal energy retention, acoustic hazard management, and exclusion of environmental moisture [Source 1].
- **Hot Air Recirculation Study**: Assess the impact of hot air recirculation on existing facility trains and vice versa [Source 2].
- **Footprint Optimization**: Evaluate Air-Cooled Heat Exchanger (ACHE) technology to reduce module size and cost, considering fan and finned tube design improvements [Source 3].
- **HVAC Design**: Ensure HVAC systems meet room temperature and humidity criteria, optimize layout for safe operation and maintenance, and locate air-cooled condensers outdoors [Source 4].

### ✅ Safety Considerations
- **Hazardous Situations**: Design must support alarming, emergency response actions, and safe release of hazardous materials [Source 1].
- **

## Score both conditions

In [8]:
baseline_score = judge_case_study(QUESTION, baseline_contexts, baseline_answer.answer)
dual_score = judge_case_study(QUESTION, dual_contexts, dual_answer.answer)

print("Baseline (Hybrid RAG only):", baseline_score.model_dump())
print()
print("Dual-Level Retrieval ALONE:", dual_score.model_dump())

Baseline (Hybrid RAG only): {'completeness': 4, 'retrieval_coverage': 4, 'groundedness': 3, 'relevance': 5, 'cross_document_integration': 5, 'reasoning': 'Completeness is scored 4 because the answer addresses engineering, safety, and environmental considerations but omits some specific details, such as the impact of ambient air temperature on cooling medium circuits mentioned in Source 2. Retrieval coverage is 4 as the contexts provide substantial evidence for most aspects but lack explicit details on certain environmental considerations like biosecurity. Groundedness is 3 because some claims, such as the appointment of an environmental manager with 10+ years of experience, are unsupported by the retrieved contexts. Relevance is 5 as the answer is tightly focused on the question without tangents. Cross-document integration is 5 because the answer synthesizes information from multiple distinct files, such as engineering considerations from Source 1 and safety considerations from Sources

## Side-by-side comparison

In [8]:
dimensions = ["completeness", "retrieval_coverage", "groundedness", "relevance", "cross_document_integration"]

print("=" * 100)
print(f"QUESTION: {QUESTION}")
print("=" * 100)
print()
print(f"{'Dimension':<30}{'Baseline (Hybrid)':<22}{'Dual-Level ALONE':<22}{'Delta'}")
print("-" * 100)
for dim in dimensions:
    b, d = getattr(baseline_score, dim), getattr(dual_score, dim)
    print(f"{dim:<30}{b:<22}{d:<22}{d - b:+d}")

print()
print("-" * 100)
print("BASELINE (Hybrid RAG only) reasoning:")
print(f"  {baseline_score.reasoning}")
print(f"  Distinct files used: {sorted(set(c.get('file_name') for c in baseline_contexts))}")
print()
print("DUAL-LEVEL ALONE reasoning:")
print(f"  {dual_score.reasoning}")
print(f"  Distinct files used: {sorted(set(c.get('file_name') for c in dual_contexts))}")
print("=" * 100)

print()
print("BASELINE ANSWER:")
print(baseline_answer.answer)
print()
print("DUAL-LEVEL ALONE ANSWER:")
print(dual_answer.answer)

QUESTION: Summarize all engineering, safety, and environmental considerations related to the Air-Cooled Condenser (ACC).

Dimension                     Baseline (Hybrid)     Dual-Level ALONE      Delta
----------------------------------------------------------------------------------------------------
completeness                  3                     3                     +0
retrieval_coverage            4                     4                     +0
groundedness                  2                     3                     +1
relevance                     4                     4                     +0
cross_document_integration    3                     3                     +0

----------------------------------------------------------------------------------------------------
BASELINE (Hybrid RAG only) reasoning:
  Completeness is scored 3 because the answer addresses engineering, safety, and environmental considerations but misses some specific details, such as hot air recirculatio

In [ ]:
RESULTS_PATH = CHATBOT_ROOT / "domain_data" / "eval_results" / "case_study_dual_level_only_vs_hybrid.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

report = {
    "question": QUESTION,
    "baseline": {
        "answer": baseline_answer.answer,
        "n_contexts": len(baseline_contexts),
        "distinct_files": sorted(set(c.get("file_name") for c in baseline_contexts)),
        "score": baseline_score.model_dump(),
    },
    "dual_level_only": {
        "answer": dual_answer.answer,
        "n_contexts": len(dual_contexts),
        "distinct_files": sorted(set(c.get("file_name") for c in dual_contexts)),
        "score": dual_score.model_dump(),
    },
}
RESULTS_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote case study report -> {RESULTS_PATH}")